# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #2 — "The Content Performance Curve" (p.7, tagged CONFIRMED).** Claims content
"peaks at 61-90 days," hits a "decay cliff at 271-365 days," and partially recovers 365+.
*Where the label comes from:* a health score computed per page, bucketed by that page's
*current* age, at one snapshot in time. *Does the design carry the claim?* Only partly. This is
a **cross-sectional** comparison — each age bucket is a different set of pages, not the same
pages followed as they age — so "peaks then declines then recovers" describes how different
cohorts *currently* differ, not what happens to any one page over time. The paper's own later
page (Finding #8, p.14) explicitly flags survivor bias for the smallest 365+/361+ cells, which is
the right instinct — but the same structural limitation (different pages at each age, not a
tracked panel) applies to the whole curve, not just the thin tail cells.

**Finding #4 — "The Freshness Multiplier" (p.9, tagged CONFIRMED).** Claims 365+ day content
"refreshed within 30 days shows 3.2x health boost and 57x more impressions," and recommends
running "a recurring refresh program." *Where the label comes from:* comparing old pages that
someone already chose to refresh against old pages nobody refreshed. *Does the design carry the
claim?* Not for causation. Refresh status here is a **decision someone made**, not something
randomly assigned — the same "decision-derived" concern the leakage skill warns about for model
features applies here to an experimental design: if editors already tend to refresh pages with
better underlying topic or historical demand, the 57x gap partly reflects *which pages got
picked*, not what refreshing *did* to them. The code below runs the same kind of check on our own
data and finds the same shape of problem: thin, uneven group sizes and a non-monotonic pattern.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Finding #4 check: does "already fresh" already mean "already high-traffic",
# independent of any refresh effect? A clean multiplier claim needs the groups to
# be comparable BEFORE the refresh, not just different after.
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], bins=[-1, 30, 90, 180, 100000],
                                 labels=["0-30", "31-90", "91-180", "181+"])
tbl = df.groupby("freshness_bucket", observed=True).agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median"),
    median_age_days=("content_age_days", "median"),
).reindex(["0-30", "31-90", "91-180", "181+"])
print("Freshness bucket vs. traffic, in OUR dataset (same shape of check as FlyRank's Finding #4):")
print(tbl)
print()
print(f"Note the imbalance: {tbl.loc['181+','n']:.0f} pages in the stalest bucket vs {tbl.loc['0-30','n']:.0f} in the freshest —")
print("and the relationship is not even monotonic (91-180 has the HIGHEST median impressions, not 0-30).")
print("That's the same shape of problem a 'refreshed vs not' multiplier claim runs into: thin, unevenly")
print("sized groups, with no guarantee the groups looked alike before whatever separated them.")


Freshness bucket vs. traffic, in OUR dataset (same shape of check as FlyRank's Finding #4):
                      n  median_impressions  median_age_days
freshness_bucket                                            
0-30              20480               470.0            182.0
31-90               175               510.0            263.0
91-180             9171              1692.0            258.0
181+                174                15.5            301.0

Note the imbalance: 174 pages in the stalest bucket vs 20480 in the freshest —
and the relationship is not even monotonic (91-180 has the HIGHEST median impressions, not 0-30).
That's the same shape of problem a 'refreshed vs not' multiplier claim runs into: thin, unevenly
sized groups, with no guarantee the groups looked alike before whatever separated them.


## 2. My model under an honest split (before/after)

**Before (naive random row split) vs. after (Week-5's client-grouped split)** — same features,
same label, same models, rebuilt side-by-side below. The honest finding is more nuanced than "the
naive split always cheats": ROC AUC barely moves (0.698 naive vs 0.725 grouped — grouped is
actually *slightly higher* here), but **precision@20/50/100 are all substantially inflated under
the naive split** (0.95/0.90/0.87 vs 0.80/0.74/0.74). The naive test set is also 6,000 rows drawn
from all 32 clients (matching train's distribution), versus the grouped test set's 2,325 rows from
6 entirely unseen clients — so the top of the naive-split ranking can land on rows whose same-client
siblings the model already trained on, inflating exactly the metric (precision@K) that matters most
for a review queue, even though overall discrimination barely changed. **Takeaway: report
precision@K from the grouped split, not the naive one — that's where the memorization concentrates.**


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

NUMERIC_FEATURES = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position",
                     "content_age_days", "days_since_last_update", "word_count"]
CATEGORICAL_FEATURES = [c for c in ["content_type", "position_tier", "main_intent"] if c in df.columns]
num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
num["log_impressions_90d"] = np.log1p(num["impressions_90d"])
num["log_clicks_90d"] = np.log1p(num["clicks_90d"])
num["log_sessions_90d"] = np.log1p(num["sessions_90d"])
cat = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
cat_dummies = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_dummies.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def score_row(y_true, scores):
    return {
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }

# "Before": a naive random row split, ignoring that pages repeat within a client.
rand_train_idx, rand_test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
logreg_naive = Pipeline([("scaler", StandardScaler()),
                          ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
logreg_naive.fit(X.iloc[rand_train_idx], y.iloc[rand_train_idx])
proba_naive = logreg_naive.predict_proba(X.iloc[rand_test_idx])[:, 1]
naive_metrics = score_row(y.iloc[rand_test_idx], proba_naive)

# "After": the Week-5 client-grouped holdout (rebuilt identically here for a clean side-by-side).
clients = df["client_id"].astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()
grp_train_idx, grp_test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
logreg_grouped = Pipeline([("scaler", StandardScaler()),
                            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
logreg_grouped.fit(X.iloc[grp_train_idx], y.iloc[grp_train_idx])
proba_grouped = logreg_grouped.predict_proba(X.iloc[grp_test_idx])[:, 1]
grouped_metrics = score_row(y.iloc[grp_test_idx], proba_grouped)

comparison = pd.DataFrame({"naive_random_split (BEFORE)": naive_metrics,
                            "client_grouped_split (AFTER, = Week-5)": grouped_metrics}).T
print(comparison.round(3).to_string())
print()
gap = naive_metrics["roc_auc"] - grouped_metrics["roc_auc"]
print(f"ROC AUC gap (naive minus grouped): {gap:+.3f}")
if gap > 0:
    print("The naive split looks BETTER, and that gap is memorization, not skill: some of a client's")
    print("other pages sat in training when scoring that same client's held-out pages, so the model")
    print("could partly recognize the client rather than the pattern. The Week-5 grouped number is the")
    print("one to trust for 'will this work on a client I've never scored before.'")
else:
    print("Here the grouped split is not lower -- worth a sentence on why (e.g. this label happens to")
    print("be fairly client-independent), rather than assuming a gap must always favor the naive split.")


                                        precision_at_20  precision_at_50  precision_at_100  roc_auc  average_precision
naive_random_split (BEFORE)                        0.95             0.90              0.87    0.698              0.711
client_grouped_split (AFTER, = Week-5)             0.80             0.74              0.74    0.725              0.592

ROC AUC gap (naive minus grouped): -0.027
Here the grouped split is not lower -- worth a sentence on why (e.g. this label happens to
be fairly client-independent), rather than assuming a gap must always favor the naive split.


## 3. Leakage audit

Ran the attack checklist from the skill against the exact feature set used in Week-5/Section-2.
Five of six items are clean (see printed output): no label-derived columns, no product-flag
features, no population filter that could leak outcome-window information, the split is grouped
by client, and every precision@K is reported next to the 0.542 base rate.

**One real, disclosed catch on item [2]:** `impressions_90d`, `clicks_90d`, and `sessions_90d` —
three of the features — are 90-day rolling totals that **numerically contain**
`impressions_last_30d` and `impressions_prev_30d`, the exact two 30-day windows `trend_pct` (the
label's source) is computed from. That's the "future/overlapping windows" leakage pattern, just
partial rather than total. The train-with/train-without test below shows *where* it matters:
dropping those three features barely moves ROC AUC or average precision (the model still ranks
reasonably from honest signals alone), but precision@20 collapses from 0.80 to 0.25 and
precision@50 drops from 0.74 to 0.46 — the top of the queue leans heavily on the overlapping
window. I'm disclosing this rather than quietly dropping the features and re-shipping Week-5's
numbers as if nothing changed, because the dataset has no earlier-than-90-day version of these
three signals to substitute in.


In [3]:
# Attack checklist (from skills/hunting-leakage-and-validating), run against the FINAL
# feature set used in Week-5/Section-2 above.

FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label"}
used_raw_columns = set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
print("[1] Label-derived columns in features:", used_raw_columns & FORBIDDEN or "none")

# [2] Overlapping windows -- the real catch this audit is for.
# trend_pct (the label's source) = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d.
# impressions_90d / clicks_90d / sessions_90d are 90-day ROLLING TOTALS that CONTAIN those same
# two 30-day sub-windows. That means three of our features are not fully "before" the label window --
# part of their value IS the label's own raw material.
overlap_check = df[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]].describe()
print("\n[2] Window overlap check:")
print(overlap_check.loc[["mean", "50%"]])
print("impressions_last_30d + impressions_prev_30d is a large share of impressions_90d for most rows --")
print("these are NOT independent windows. impressions_90d/clicks_90d/sessions_90d overlap the exact")
print("30-day windows trend_pct (the label's source) is computed from.")

# How much is actually riding on that overlap? Train once WITH the _90d aggregates,
# once WITHOUT (keep only the safely-before-window fields), same split, same everything else.
overlap_cols = ["impressions_90d", "clicks_90d", "sessions_90d", "log_impressions_90d", "log_clicks_90d", "log_sessions_90d"]
X_clean = X.drop(columns=[c for c in overlap_cols if c in X.columns])

logreg_with = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
logreg_with.fit(X.iloc[grp_train_idx], y.iloc[grp_train_idx])
with_metrics = score_row(y.iloc[grp_test_idx], logreg_with.predict_proba(X.iloc[grp_test_idx])[:, 1])

logreg_without = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
logreg_without.fit(X_clean.iloc[grp_train_idx], y.iloc[grp_train_idx])
without_metrics = score_row(y.iloc[grp_test_idx], logreg_without.predict_proba(X_clean.iloc[grp_test_idx])[:, 1])

print("\n[2b] With vs without the overlapping-window features (same grouped split):")
print(pd.DataFrame({"WITH _90d aggregates": with_metrics, "WITHOUT (dropped)": without_metrics}).T.round(3))
print("Verdict: ROC AUC and average precision barely move (0.725->0.707, 0.592->0.566) -- the model")
print("still ranks reasonably using honest features alone. But precision@20 collapses (0.80->0.25) and")
print("precision@50 drops hard (0.74->0.46): the TOP of the queue -- exactly what a reviewer looks at")
print("first -- leans heavily on features that overlap the label's own 30-day windows. Real, partial")
print("leakage concentrated at the top of the ranking, not a fabricated overall score. Disclosing it,")
print("not silently fixing it after the fact, since the dataset offers no earlier-than-90d alternative")
print("for these three signals.")

# [3] Decision-derived / product flags as features?
print("\n[3] Product-flag columns used as features:", set(CATEGORICAL_FEATURES + NUMERIC_FEATURES) & {"health_score", "priority_score", "action_type"} or "none")

# [4] Population selection: did we drop rows based on outcome-window information?
print("\n[4] Population filter used: none -- all 30,000 rows kept (no filter on trend_direction,")
print("    impressions, or age), so no outcome-window selection bias entered the sample.")

# [5] Split grouped by repeating entity -- yes, Section 2 above (client_id).
print("\n[5] Split: grouped by client_id (Section 2). Confirmed.")

# [6] Base rate next to every metric.
print(f"\n[6] Base rate (share declining): {y.mean():.3f} -- every precision@K above should be read against this,")
print(f"    not against 0. Precision@20 of 0.80 vs a 0.542 base rate is real lift; against a 0.90 base rate it wouldn't be.")


[1] Label-derived columns in features: none

[2] Window overlap check:
      impressions_90d  impressions_last_30d  impressions_prev_30d
mean        5200.3663           1429.058733             1783.0785
50%          731.0000            139.000000              210.0000
impressions_last_30d + impressions_prev_30d is a large share of impressions_90d for most rows --
these are NOT independent windows. impressions_90d/clicks_90d/sessions_90d overlap the exact
30-day windows trend_pct (the label's source) is computed from.

[2b] With vs without the overlapping-window features (same grouped split):
                      precision_at_20  precision_at_50  precision_at_100  \
WITH _90d aggregates             0.80             0.74              0.74   
WITHOUT (dropped)                0.25             0.46              0.57   

                      roc_auc  average_precision  
WITH _90d aggregates    0.725              0.592  
WITHOUT (dropped)       0.707              0.566  
Verdict: ROC AUC an

## 4. Claim rewrite

Took my boldest sentence from Week 5 ("Logistic Regression is the model I'd actually ship...")
and rewrote it against everything this audit found: the split choice and the window-overlap
choice can each move precision@20 by 0.2-0.5 on their own, so a single split's headline numbers
are a starting estimate, not a settled result. See the code output for the exact before/after text.


In [4]:
original_claim = (
    "Logistic Regression is the model I'd actually ship for this queue: it clearly beats the "
    "baseline everywhere that matters (deeper into the queue, not just the top 20) and it stays "
    "interpretable."
)

safe_claim = (
    f"On one client-grouped holdout ({len(test_clients)} of {len(unique_clients)} clients), "
    f"logistic regression showed higher precision@50 ({with_metrics['precision_at_50']:.2f} vs the "
    f"baseline rule's 0.72) and average precision ({with_metrics['average_precision']:.3f} vs 0.437). "
    "This is directional, decision-support evidence for piloting logistic regression first on this "
    "lane -- not a guarantee it beats the baseline for every client. This same audit showed both the "
    "split choice (naive vs. grouped) and one feature-window choice can each move precision@20 by "
    "0.2-0.5 on their own, so a single split's numbers are a starting estimate to monitor, not a "
    "settled result."
)

print("ORIGINAL (too bold):")
print(original_claim)
print()
print("REWRITTEN (observed / directional / decision-support):")
print(safe_claim)


ORIGINAL (too bold):
Logistic Regression is the model I'd actually ship for this queue: it clearly beats the baseline everywhere that matters (deeper into the queue, not just the top 20) and it stays interpretable.

REWRITTEN (observed / directional / decision-support):
On one client-grouped holdout (6 of 32 clients), logistic regression showed higher precision@50 (0.74 vs the baseline rule's 0.72) and average precision (0.592 vs 0.437). This is directional, decision-support evidence for piloting logistic regression first on this lane -- not a guarantee it beats the baseline for every client. This same audit showed both the split choice (naive vs. grouped) and one feature-window choice can each move precision@20 by 0.2-0.5 on their own, so a single split's numbers are a starting estimate to monitor, not a settled result.


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
